# Who was in the treatment arm, six months later

Somebody asks. The spreadsheet the analyst used has been re-exported, the randomizer was a
notebook cell with `rng = default_rng()` and no seed, and the person who ran it has changed
teams. What survives is a column of arm labels that nobody can re-derive, which means nobody
can tell a randomization that worked from one that was overwritten by a filter three steps
downstream.

`match_clusters` pairs clusters and flips a coin inside each pair — that is a design
calculation, and it has been here since Phase 5. This notebook is the other half: the thing
that runs on the day, and can be asked afterwards to prove what it did.

In [ ]:
import numpy as np

from axiom.core import Unit
from axiom.design import (
    RERANDOMIZED, ArmAllocation, ArmAssignment, AssignMethod, Assigned, BalanceRow,
    arm_for, assign, bucket, standardized_differences,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way

enable();  # every axiom result renders itself

units = tuple(f"u{i:05d}" for i in range(4000))
split: ArmAllocation = ArmAllocation.equal("control", "treated")
print(split.arms, split.shares, "| reference arm:", split.reference)

## The arm is a function, not a record

`bucket` hashes the unit id with a salt into a uniform draw, and `arm_for` reads the arm off
it. No roster, no state, no ordering — which is what makes it re-derivable. Two systems that
never talk agree; a unit that first appears in week three lands where it always would have.

In [ ]:
draws = np.asarray([bucket(u, salt="NW-14") for u in units])
print(f"uniform: min {draws.min():.6f}, max {draws.max():.6f} (< 1 by construction),"
      f" mean {draws.mean():.4f}")

week_one = assign(units[:200], split, salt="NW-14")
week_three = assign(units, split, salt="NW-14")
same = all(week_one.arm(u) == week_three.arm(u) for u in units[:200])
print("the first 200 units did not move when 3800 more arrived:", same)
print("a unit nobody had seen:", arm_for("signed-up-on-tuesday", split, salt="NW-14"))
rerun = assign(units, split, salt="rerun")
moved = int(np.count_nonzero(week_three.arm_of != rerun.arm_of))
print(f"changing the salt is a fresh randomization: {moved} of {len(units)} units move")

## What the hash costs, and what a block buys

A hash split is binomial, so the realized shares are only approximately the target ones. When
the split has to be exact — a small study, or a three-arm design where 30 units in the wrong
arm is a power problem — `method="block"` fills permuted blocks within each stratum instead.
It needs the roster and a seed, and records both.

In [ ]:
tri = ArmAllocation(arms=("control", "low", "high"), shares=(0.5, 0.25, 0.25))
rows = []
for method in ("hash", "block"):
    kwargs = {"salt": "NW-14"} if method == "hash" else {"seed": 4}
    result = assign(units, tri, method=method, **kwargs)
    realized = ", ".join(f"{a} {result.spec.share_of(a):.4f}" for a in tri.arms)
    rows.append([method, str(result.counts()), realized, result.spec.block or "—"])
table(rows, headers=("method", "counts", "realized shares", "block size"),
      title=f"target shares {tri.shares}")
print("smallest block with whole counts:", tri.block_size(), "| bucket edges:", tri.edges())

## Strata, and balance on what you recorded

`strata` blocks within each label, so the split is exact inside every stratum rather than only
overall. `standardized_differences` is the balance table — arm means and the worst
standardized gap — and it is a description of one draw, not a proof that the draw was
exchangeable.

In [ ]:
rng = np.random.default_rng(0)
region = ["north" if i % 2 else "south" for i in range(len(units))]
covariates = {"age": rng.uniform(20, 60, len(units)), "pre_outcome": rng.normal(size=len(units))}

stratified = assign(units, split, method="block", seed=4, strata=region, covariates=covariates)
for label in ("north", "south"):
    members = [i for i, s in enumerate(region) if s == label]
    treated = int(np.count_nonzero(stratified.arm_of[members] == 1))
    print(f"{label}: {treated} treated of {len(members)} — exact within the stratum")

rows: list[BalanceRow] = list(stratified.spec.balance)
table([[r.covariate, f"{r.means[0]:.4f}", f"{r.means[1]:.4f}", f"{r.smd:+.4f}"] for r in rows],
      headers=("covariate", "control", "treated", "smd"),
      title=f"balance, worst standardized difference {stratified.spec.worst_smd:.4f}")

## Re-randomizing, and the thing people forget it costs

`method="rerandomize"` draws until the worst standardized difference is under a threshold. It
works, and it changes the design: the estimator's sampling distribution is narrower than the
complete-randomization one a Wald interval assumes, so **the interval is conservative**.
`RERANDOMIZED` rides on the result saying so, and names the two analyses that recover the
power.

In [ ]:
tight = assign(units, split, method="rerandomize", seed=4, strata=region,
               covariates=covariates, threshold=0.005, max_draws=400)
print(f"threshold met in {tight.spec.draws_used} draws: {tight.spec.balance_met}"
      f" | worst smd {stratified.spec.worst_smd:.4f} -> {tight.spec.worst_smd:.4f}")

line = tight.spec.ledger_line()
print("\n" + line.statement)
print("assumption:", RERANDOMIZED.name, "|", RERANDOMIZED.state)
print("  ", RERANDOMIZED.statement)
print("  challenged by:", RERANDOMIZED.challenged_by)

impossible = assign(units[:40], split, method="rerandomize", seed=4,
                    covariates={"age": covariates["age"][:40]}, threshold=1e-9, max_draws=25)
print("\nan unreachable threshold:", impossible.spec.balance_met,
      "—", impossible.spec.ledger_line().statement[-70:])

## The audit

An `Assigned` carries the arms as an array; its `spec` is an `ArmAssignment` — the *rule*,
plus the counts, the balance table and a content hash of the roster. The rule is a `Spec`, so
it hashes and round-trips and travels; the roster does not go into it. `verify()` puts the two
back together, re-derives every arm, and answers in the identification vocabulary rather than
with a boolean.

In [ ]:
methods: list[AssignMethod] = ["hash", "block", "rerandomize"]
verdicts = []
for method in methods:
    kwargs = {"salt": "NW-14"} if method == "hash" else {"seed": 4}
    if method == "rerandomize":
        kwargs |= {"covariates": covariates, "threshold": 0.01, "max_draws": 300}
    result = assign(units, split, method=method, **kwargs)
    verdicts.append([method, result.verify().status, result.spec.roster_hash[:12] + "…",
                     result.spec.content_hash()[:12] + "…"])
table(verdicts, headers=("method", "re-derives to", "roster hash", "rule hash"))

tampered = Assigned(spec=week_three.spec, units=week_three.units,
                    arm_of=np.where(np.arange(len(units)) == 7, 1 - week_three.arm_of, week_three.arm_of))
verdict = tampered.verify()
print("\none unit moved by hand:", verdict.status)
print(" ", verdict.reason)

rule = ArmAssignment.from_json(week_three.spec.to_json())
print("\nthe rule round-trips:", rule == week_three.spec,
      "| carries the roster:", "u00000" in week_three.spec.to_json())

## What this notebook decided

- The arm a unit is in should be a **function of its id**, not a row somebody kept. Hash
  assignment is stateless, order-independent, and re-derivable by anyone with the salt; its
  price is that the split is only approximately the target one.
- When the split has to be exact, blocks within strata buy that with a roster and a seed, both
  recorded on the rule.
- Re-randomization buys balance and changes the design. The result says so, in an `Assumption`
  with a `challenged_by` naming what recovers the power.
- The rule travels; the roster does not. `verify()` is the difference between an assignment
  somebody remembers and one anybody can check.

`nbs/diagnose/07-delivery.ipynb` takes the next step: the units that actually turned up are
not always the ones the rule assigned, and that gap is checkable too.